In [1]:
import sys, random
sys.path.insert(0, "..")
import torch
from sorl.trainer_ablate import _drop_nl_prefix_m_set

# ── Fake vocabulary layout (mirrors SoRL) ──────────────────────────────────
#   0             = PAD
#   1..BASE_VOCAB-1 = NL tokens   (820 = answer delimiter "####")
#   BASE_VOCAB    = placeholder abstract token (unused by real tokens)
#   BASE_VOCAB+1+ = actual abstract tokens
BASE_VOCAB = 1000
PAD_ID     = 0
ANSWER_ID  = 820   # same default as get_answer_start_index / _drop_nl_prefix_m_set

def tok_name(t):
    t = int(t)
    if t == PAD_ID:    return "[PAD]"
    if t == ANSWER_ID: return "[####]"
    if t >= BASE_VOCAB: return f"[A{t-BASE_VOCAB}]"
    return f"nl{t}"

def show_seq(seq_1d, label=""):
    toks = [tok_name(t) for t in seq_1d if int(t) != PAD_ID]
    print(f"  {label:25s}  {' '.join(toks)}")

print("Imports OK")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Imports OK


In [2]:
def make_sorl_seq(q_len=3, cot_nl=16, K=4, ans_nl=2):
    """
    Build a synthetic SoRL-expanded sequence.
    Layout: [Q]*q_len  [cot_nl NL tokens, one abstract per K NL]  [####]  [answer]*ans_nl
    Abstract tokens are BASE_VOCAB+1, BASE_VOCAB+2, ...
    """
    q   = list(range(10, 10 + q_len))                        # question NL tokens
    cot = []
    abs_id = BASE_VOCAB + 1
    for i in range(cot_nl):
        cot.append(100 + i)                                   # NL CoT token
        if (i + 1) % K == 0:
            cot.append(abs_id)                                # abstract token every K NL
            abs_id += 1
    ans = [ANSWER_ID] + list(range(200, 200 + ans_nl))        # #### ans0 ans1 ...
    return torch.tensor(q + cot + ans, dtype=torch.long)

# ── Build a batch of two sequences ────────────────────────────────────────
seq0 = make_sorl_seq(q_len=3, cot_nl=16, K=4, ans_nl=2)   # 16 NL CoT → 4 abstract tokens
seq1 = make_sorl_seq(q_len=3, cot_nl=8,  K=4, ans_nl=2)   # 8  NL CoT → 2 abstract tokens

L    = max(len(seq0), len(seq1))
ids  = torch.full((2, L), PAD_ID, dtype=torch.long)
attn = torch.zeros(2, L, dtype=torch.long)
ids[0, :len(seq0)] = seq0;  attn[0, :len(seq0)] = 1
ids[1, :len(seq1)] = seq1;  attn[1, :len(seq1)] = 1
prompt_len = torch.tensor([3, 3])

print("=== ORIGINAL SEQUENCES ===")
show_seq(ids[0], f"batch[0]  len={len(seq0)}")
show_seq(ids[1], f"batch[1]  len={len(seq1)}")

=== ORIGINAL SEQUENCES ===
  batch[0]  len=26           nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 [A4] [####] nl200 nl201
  batch[1]  len=16           nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] [####] nl200 nl201


In [ ]:
random.seed(0)

M_SETS = {
    "m=0 only (no compression)":   (0,),
    "m in {0,8}":                   (0, 8),
    "M_SET=[0,16,32,64,128]":       (0, 16, 32, 64, 128),
}

for label, m_set in M_SETS.items():
    print(f"\n{'─'*70}")
    print(f"  {label}  →  sampled per batch item independently")
    print(f"{'─'*70}")
    for trial in range(3):
        s, out_attn, out_pl = _drop_nl_prefix_m_set(
            ids, attn, prompt_len, BASE_VOCAB, PAD_ID, m_set=m_set
        )
        print(f"  trial {trial+1}:")
        show_seq(out[0], f"  batch[0]  len={int(out_attn[0].sum())}")
        show_seq(out[1], f"  batch[1]  len={int(out_attn[1].sum())}")


──────────────────────────────────────────────────────────────────────
  m=0 only (no compression)  →  sampled per batch item independently
──────────────────────────────────────────────────────────────────────
  trial 1:
    batch[0]  len=26         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 [A4] [####] nl200 nl201
    batch[1]  len=16         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] [####] nl200 nl201
  trial 2:
    batch[0]  len=26         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 [A4] [####] nl200 nl201
    batch[1]  len=16         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] [####] nl200 nl201
  trial 3:
    batch[0]  len=26         nl10 nl11 nl12 nl100 nl101 nl102 nl103 [A1] nl104 nl105 nl106 nl107 [A2] nl108 nl109 nl110 nl111 [A3] nl112 nl113 nl114 nl115 

## Real pipeline: GSM8K + SoRL search → `_drop_nl_prefix_m_set`

In [5]:
import os, sys
sys.path.insert(0, "..")
import torch
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.sorl_trainer import sorl_search
from sorl.trainer_ablate import _drop_nl_prefix_m_set
from data.pt_dataset import get_dataset, collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

MODEL_NAME  = "Qwen/Qwen3-0.6B"
ABS_VOCAB   = 32
K           = 4
MAX_LENGTH  = 256
N_SAMPLES   = 4   # number of GSM8K samples to show

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[ABS_VOCAB])
model     = model.to(device).eval()

base_vocab  = int(model.vocab_sizes[0].item())
pad_id      = tokenizer.pad_token_id
print(f"base_vocab={base_vocab}  abs_vocab={ABS_VOCAB}  pad_id={pad_id}")

device: cpu


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


base_vocab=151936  abs_vocab=32  pad_id=151643


In [ ]:
train_ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=MAX_LENGTH)

# grab N_SAMPLES items and collate into a batch
samples  = [train_ds[i] for i in range(N_SAMPLES)]
batch    = collate_fn(samples)

input_ids  = batch["input_ids"].to(device)
attn_mask  = batch["attention_mask"].to(device)
prompt_len = batch["prompt_len"].to(device)

# Not exactly correct: 
# - "ans" contains both CoT and Answer

print("=== RAW GSM8K SEQUENCES ===")
for b in range(N_SAMPLES):
    valid = int(attn_mask[b].sum())
    text  = tokenizer.decode(input_ids[b, :valid], skip_special_tokens=False)
    pl    = prompt_len[b].item()
    q     = tokenizer.decode(input_ids[b, :pl],     skip_special_tokens=True)
    ans   = tokenizer.decode(input_ids[b, pl:valid], skip_special_tokens=True)
    print(f"\n[{b}] len={valid}  prompt_len={pl}")
    print(f"  Q  : {q[:120]}")
    print(f"  CoT: {ans[:]}")

=== RAW GSM8K SEQUENCES ===

[0] len=100  prompt_len=42
  Q  : Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips 
  CoT:  Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

[1] len=99  prompt_len=35
  Q  : Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she ea
  CoT:  Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

[2] len=164  prompt_len=64
  Q  : Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her paren
  CoT:  In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5

[3] len=183  prompt_len=57
  Q  : Question

In [8]:
# Run sorl_search → get best_data with interleaved abstract tokens
with torch.no_grad():
    best_data, best_ppt, best_ppt_adv, expanded_attn, expanded_pl = sorl_search(
        model, input_ids, attn_mask, prompt_len, pad_id,
        n=2, K=K, max_iterations=2,
        memory_span_abs=1792, memory_span_traj=1792,
        temperature=1.0,
    )

def decode_annotated(ids_1d, attn_1d, base_vocab, tokenizer):
    """Decode sequence: NL tokens → text, abstract tokens → [ABS], answer delimiter preserved."""
    valid = int(attn_1d.sum())
    parts = []
    buf   = []
    for tid in ids_1d[:valid].tolist():
        if tid >= base_vocab:
            if buf:
                parts.append(tokenizer.decode(buf, skip_special_tokens=False))
                buf = []
            parts.append(f"[ABS]")
        else:
            buf.append(tid)
    if buf:
        parts.append(tokenizer.decode(buf, skip_special_tokens=False))
    return "".join(parts)

print("=== AFTER sorl_search (interleaved abstract tokens) ===")
for b in range(N_SAMPLES):
    pl   = expanded_pl[b].item()
    text = decode_annotated(best_data[b], expanded_attn[b], base_vocab, tokenizer)
    q    = tokenizer.decode(best_data[b, :pl].tolist(), skip_special_tokens=True)
    print(f"\n[{b}]  expanded_len={int(expanded_attn[b].sum())}  prompt_len={pl}")
    print(f"  {text[:350]}")

=== AFTER sorl_search (interleaved abstract tokens) ===

[0]  expanded_len=124  prompt_len=52
  Question: Natalia[ABS] sold clips to [ABS]48 of her[ABS] friends in April,[ABS] and then she sold[ABS] half as many clips[ABS] in May. How[ABS] many clips did Natal[ABS]ia sell altogether in[ABS] April and May?
[ABS]Answer: Natalia[ABS] sold 48[ABS]/2 = <<[ABS]48/2[ABS]=24>>[ABS]24 clips in[ABS] May.
Natal[ABS]ia sold 4[ABS]8+24[ABS] = <<48[ABS]+24

[1]  expanded_len=123  prompt_len=43
  Question: Weng[ABS] earns $12[ABS] an hour for babys[ABS]itting. Yesterday,[ABS] she just did [ABS]50 minutes of[ABS] babysitting. How[ABS] much did she earn[ABS]?
Answer: W[ABS]eng earns 1[ABS]2/60[ABS] = $<<1[ABS]2/60[ABS]=0.2[ABS]>>0.2[ABS] per minute.
Working[ABS] 50 minutes[ABS], she earned [ABS]0.2 x[ABS] 50 =[ABS] $<<0.[ABS]2*50[ABS]=10>>[AB

[2]  expanded_len=204  prompt_len=79
  Question: Betty is[ABS] saving money for a[ABS] new wallet which costs[ABS] $100[ABS]. Betty has only[ABS] half of the mon

In [9]:
import random
random.seed(42)

M_SET = (0, 16, 32, 64, 128)

print(f"=== AFTER _drop_nl_prefix_m_set  (M_SET={M_SET}) ===")
print("CoT prefix NL tokens are dropped; their abstract tokens are kept.")
print("'####' and the answer are always preserved.\n")

for trial in range(3):
    print(f"{'─'*70}  trial {trial+1}")
    out_ids, out_attn, out_pl = _drop_nl_prefix_m_set(
        best_data, expanded_attn, expanded_pl,
        base_vocab, pad_id, m_set=M_SET,
        answer_token_id=820,
    )
    for b in range(N_SAMPLES):
        text = decode_annotated(out_ids[b], out_attn[b], base_vocab, tokenizer)
        delta = int(expanded_attn[b].sum()) - int(out_attn[b].sum())
        print(f"  [{b}]  len {int(expanded_attn[b].sum())} → {int(out_attn[b].sum())}  (−{delta} NL tokens dropped)")
        print(f"       {text[:300]}")
    print()

=== AFTER _drop_nl_prefix_m_set  (M_SET=(0, 16, 32, 64, 128)) ===
CoT prefix NL tokens are dropped; their abstract tokens are kept.
'####' and the answer are always preserved.

──────────────────────────────────────────────────────────────────────  trial 1
  [0]  len 124 → 124  (−0 NL tokens dropped)
       Question: Natalia[ABS] sold clips to [ABS]48 of her[ABS] friends in April,[ABS] and then she sold[ABS] half as many clips[ABS] in May. How[ABS] many clips did Natal[ABS]ia sell altogether in[ABS] April and May?
[ABS]Answer: Natalia[ABS] sold 48[ABS]/2 = <<[ABS]48/2[ABS]=24>>[ABS]24 clips in[ABS] May
  [1]  len 123 → 123  (−0 NL tokens dropped)
       Question: Weng[ABS] earns $12[ABS] an hour for babys[ABS]itting. Yesterday,[ABS] she just did [ABS]50 minutes of[ABS] babysitting. How[ABS] much did she earn[ABS]?
Answer: W[ABS]eng earns 1[ABS]2/60[ABS] = $<<1[ABS]2/60[ABS]=0.2[ABS]>>0.2[ABS] per minute.
Working[ABS] 50 minutes[ABS], she earned [AB
  [2]  len 204 → 172  (−32 NL tokens 

## `cot_only_abs` vs default: insertion mask comparison on real GSM8K sequences

In [ ]:
from sorl.sorl_trainer import infer_insert_mask, insert_tokens_with_padding

# Re-use the batch from cells 5-6 (input_ids, attn_mask, prompt_len, model, tokenizer, base_vocab, pad_id)
# K=4 as configured above


def compare_insertion(ids, attn, pl, K, base_vocab, pad_id, tokenizer, model, label_a, label_b):
    """Show abstract token insertion positions side-by-side for default vs cot_only_abs."""
    ANSWER_ID = 820

    mask_default = infer_insert_mask(ids, K, attn, prompt_len=pl)                          # full seq
    mask_cot     = infer_insert_mask(ids, K, attn, prompt_len=pl, answer_token_id=ANSWER_ID)  # CoT only

    exp_default, exp_attn_d = insert_tokens_with_padding(ids, attn, mask_default, model.vocab_sizes[0], pad_id)
    exp_cot,     exp_attn_c = insert_tokens_with_padding(ids, attn, mask_cot,     model.vocab_sizes[0], pad_id)

    print(f"{'─'*80}")
    for b in range(ids.shape[0]):
        vd = int(exp_attn_d[b].sum())
        vc = int(exp_attn_c[b].sum())
        print(f"\n  sample [{b}]   {label_a}: {vd} tokens   {label_b}: {vc} tokens")
        print(f"  {label_a}: {decode_annotated(exp_default[b], exp_attn_d[b], base_vocab, tokenizer)[:320]}")
        print(f"  {label_b}: {decode_annotated(exp_cot[b],     exp_attn_c[b], base_vocab, tokenizer)[:320]}")

compare_insertion(input_ids, attn_mask, prompt_len, K, base_vocab, pad_id, tokenizer, model,
                  "default (full-seq)", "cot_only_abs")

TypeError: infer_insert_mask() got an unexpected keyword argument 'answer_token_id'

In [ ]:
from sorl.sorl_trainer import sorl_search

# Run sorl_search with and without cot_only_abs, show decoded best sequences
with torch.no_grad():
    best_default, _, _, attn_def, pl_def = sorl_search(
        model, input_ids, attn_mask, prompt_len, pad_id,
        n=2, K=K, max_iterations=2,
        memory_span_abs=1792, memory_span_traj=1792,
        temperature=1.0,
        response_only_abs=True,   # restrict to response
        cot_only_abs=False,
    )
    best_cot, _, _, attn_cot, pl_cot = sorl_search(
        model, input_ids, attn_mask, prompt_len, pad_id,
        n=2, K=K, max_iterations=2,
        memory_span_abs=1792, memory_span_traj=1792,
        temperature=1.0,
        response_only_abs=False,
        cot_only_abs=True,        # CoT only — no abs in answer
    )

print("=== sorl_search: response_only_abs=True (abs in CoT + answer) ===")
for b in range(N_SAMPLES):
    text = decode_annotated(best_default[b], attn_def[b], base_vocab, tokenizer)
    print(f"  [{b}] len={int(attn_def[b].sum())}  {text[:300]}")

print("\n=== sorl_search: cot_only_abs=True (abs in CoT only, NOT in answer) ===")
for b in range(N_SAMPLES):
    text = decode_annotated(best_cot[b], attn_cot[b], base_vocab, tokenizer)
    print(f"  [{b}] len={int(attn_cot[b].sum())}  {text[:300]}")